In [15]:
##### Assembles rasters on tonnes of livestock products and overall sum of livestock production 
# Distributes FAO country-level production stats to pixels based on livestock density  
# Intermediate step - convert livestock density rasters into absolute livestock rasters

import os
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.features import rasterize
from rasterio.transform import from_bounds
import shutil

In [16]:
##### Load data

# Get the current working directory
cd = os.path.dirname(os.getcwd())

# load country data 
country_total = pd.read_csv(f"{cd}/Data/Clean/Production/FAO_production_by_animal.csv")
country_shp = gpd.read_file("/Users/carinamanitius/Documents/Data/Admin_Boundaries/gadm_410-levels.gpkg", layer='ADM_0')


In [17]:
##### Define function to get total number of livestock in each pixel 

def density_to_absolute (animal_name, animal_code ):
    raster_path = f"{cd}/Data/Raw/FAO_gridded_livestock/GLW4-2020.D-DA.{animal_code}.tif"
    output_path = f"{cd}/Data/Clean/Production/Animals/{animal_name}.tif"

    # load raster
    with rasterio.open(raster_path) as src:
            density = src.read(1).astype(np.float64)   
            nodata  = src.nodata
            meta    = src.meta.copy()
            transform = src.transform
            crs     = src.crs
            height, width = density.shape

            # Aproximate the area of each pixel in m2 using lat and long of each pixel 
            res_deg = abs(transform.e)           # degrees per pixel (latitude)
            res_lon = abs(transform.a)           # degrees per pixel (longitude)
            # latitude of each row centre
            lats = np.array([
                transform.f + (r + 0.5) * transform.e
                for r in range(height)
            ])
            # area = (lon_res * lat_res) * (111.32 km/deg)² * cos(lat)
            pixel_area_km2 = (
                res_lon * res_deg *
                (111.32 ** 2) *
                np.abs(np.cos(np.radians(lats)))
            )[:, np.newaxis]  # shape (height, 1) – broadcast over columns

    if nodata is not None:
        density[density == nodata] = np.nan
    density[density < 0] = np.nan


    #### CALCULATE
    # total animals per pixel = density * area
    # density = animial / km2
    # area = km2

    output = np.full((height, width), np.nan, dtype=np.float64)
    animal_per_pixel = density * pixel_area_km2

    ##### WRITE OUTPUT 

    meta.update(
            dtype   = "float32",
            count   = 1,
            nodata  = -9999,
            compress= "lzw"
        )

    output = animal_per_pixel

    out_arr = output.astype(np.float32)
    out_arr[np.isnan(out_arr)] = -9999

    with rasterio.open(output_path, "w", **meta) as dst:
        dst.write(out_arr, 1)

    total_animals = np.nansum(animal_per_pixel)

    return total_animals

In [18]:
def process_absolute_raster(animal_name, animal_code):
    raster_path = f"{cd}/Data/Raw/FAO_gridded_livestock/GLW4-2020.D-DA.{animal_code}.tif"
    output_path = f"{cd}/Data/Clean/Production/Animals/{animal_name}.tif"

    with rasterio.open(raster_path) as src:
        data    = src.read(1).astype(np.float64)
        nodata  = src.nodata
        meta    = src.meta.copy()

    # Mask nodata and invalid values
    if nodata is not None:
        data[data == nodata] = np.nan
    data[data < 0] = np.nan

    # Match output metadata to density_to_absolute outputs
    meta.update(
        dtype   = "float32",
        count   = 1,
        nodata  = -9999,
        compress= "lzw"
    )

    out_arr = data.astype(np.float32)
    out_arr[np.isnan(out_arr)] = -9999

    with rasterio.open(output_path, "w", **meta) as dst:
        dst.write(out_arr, 1)

    total_animals = np.nansum(data)

    return total_animals

In [19]:
##### Run gridded to absolute function for animals with density data  

den_names = ['cattle', 'goats', 'sheep', 'pigs', 'buffalo', 'chicken']
den_codes = ['CTL', 'GTS', 'SHP', 'PGS', 'BFL', 'CHK']

for name, code in zip(den_names, den_codes):
    density_to_absolute(name, code)

##### Run cleaning function for animals in absolute already 
den_names = ['horses', 'ducks']
den_codes = ['HOR', 'DCK']

for name, code in zip(den_names, den_codes):
    process_absolute_raster(name, code)

In [20]:
##### Create all raster by adding all animals 

##### Create aggregate raster of total animals across all categories

all_animal_names = ['cattle', 'goats', 'sheep', 'pigs', 'buffalo', 'chicken', 'horses', 'ducks']

aggregate = None
meta = None

for animal_name in all_animal_names:
    raster_path = f"{cd}/Data/Clean/Production/Animals/{animal_name}.tif"
    
    with rasterio.open(raster_path) as src:
        data = src.read(1).astype(np.float64)
        if meta is None:
            meta = src.meta.copy()
    
    # Replace nodata with nan for summation
    data[data == -9999] = np.nan
    
    if aggregate is None:
        aggregate = np.zeros_like(data)
    
    # Add, treating nan as zero (absent data shouldn't void the whole sum)
    aggregate = np.nansum(np.stack([aggregate, data]), axis=0)

##### Write aggregate raster

output_path = f"{cd}/Data/Clean/Production/Animals/all.tif"

meta.update(
    dtype   = "float32",
    count   = 1,
    nodata  = -9999,
    compress= "lzw"
)

out_arr = aggregate.astype(np.float32)
out_arr[np.isnan(out_arr)] = -9999

with rasterio.open(output_path, "w", **meta) as dst:
    dst.write(out_arr, 1)


# save copy of all as other for use as proxy
src = f"{cd}/Data/Clean/Production/Animals/all.tif"
dst = f"{cd}/Data/Clean/Production/Animals/other.tif"

shutil.copy(src, dst)

'/Users/carinamanitius/Documents/GitHub/AgDownscaling/Data/Clean/Production/Animals/other.tif'

In [21]:
##### Define function to distribute country animal production by livestock distirbution 

def apportion_country_production (animal_name):

    ### Define
    raster_path = f"{cd}/Data/Clean/Production/Animals/{animal_name}.tif"
    output_path = f"{cd}/Data/Clean/Production/{animal_name}_production_tonnes_2020.tif"
    column_title = f"{animal_name}_t"
    country_shp = gpd.read_file("/Users/carinamanitius/Documents/Data/Admin_Boundaries/gadm_410-levels.gpkg", layer='ADM_0')

    ### Set-up 
    # create dictionary of countries 
    animal_dict = dict(zip(country_total['ISO3'], country_total[column_title]))

    # load raster 
    with rasterio.open(raster_path) as src:
            animals_per_pixel = src.read(1).astype(np.float64)  
            nodata  = src.nodata
            meta    = src.meta.copy()
            transform = src.transform
            crs     = src.crs
            height, width = animals_per_pixel.shape

    if nodata is not None:
        animals_per_pixel[animals_per_pixel == nodata] = np.nan

    # ensure crs's match
    country_shp = country_shp.to_crs(crs)

    # set output shape
    output = np.full((height, width), np.nan, dtype=np.float64)

    total_countries  = len(country_shp)

    # iterate through each country, creating a mask and reapportioning the country total based on livestock 
    for _, row in country_shp.iterrows():
        iso3 = str(row['GID_0']).upper()
        animals = animal_dict.get(iso3, None)

        if animals is None or np.isnan(animals):
            continue   

        # Rasterize country mask
        mask = rasterize(
            [(row.geometry, 1)],
            out_shape=(height, width),
            transform=transform,
            fill=0,
            dtype=np.uint8
        ).astype(bool)

        country_animals = animals_per_pixel.copy()
        country_animals[~mask] = np.nan

        total_animals = np.nansum(country_animals)

        if total_animals <= 0:
            continue  # FIX 3: leave as NaN (missing) rather than distributing uniformly
        else:
            weight = country_animals / total_animals   # fraction per pixel
            output[mask] = weight[mask] * animals

    ### Write output 

    meta.update(
        dtype   = "float32",
        count   = 1,
        nodata  = -9999,
        compress= "lzw"
    )

    out_arr = output.astype(np.float32)
    out_arr[np.isnan(out_arr)] = -9999

    with rasterio.open(output_path, "w", **meta) as dst:
        dst.write(out_arr, 1)


In [22]:
##### Run country appotionment function for all animals 

names = ['cattle', 'goats', 'sheep', 'pigs', 'horses', 'ducks', 'buffalo', 'other', 'chicken']

for name in names:
    apportion_country_production(name)

In [23]:
##### Calculate total animal production (tonnes) across all 

animal_names = ['cattle', 'goats', 'sheep', 'pigs', 'horses', 'ducks', 'buffalo', 'other', 'chicken']

# initialise sum array from first raster
with rasterio.open(f"{cd}/Data/Clean/Production/{animal_names[0]}_production_tonnes_2020.tif") as src:
    total = src.read(1).astype(np.float64)
    nodata = src.nodata
    meta = src.meta.copy()

total[total == nodata] = np.nan

# add remaining animals
for animal_name in animal_names[1:]:
    with rasterio.open(f"{cd}/Data/Clean/Production/{animal_name}_production_tonnes_2020.tif") as src:
        arr = src.read(1).astype(np.float64)
        arr[arr == src.nodata] = np.nan
    total = np.nansum(np.stack([total, arr]), axis=0)

# write output
meta.update(dtype="float32", count=1, nodata=-9999, compress="lzw")
out_arr = total.astype(np.float32)
out_arr[np.isnan(out_arr)] = -9999

with rasterio.open(f"{cd}/Data/Clean/Production/livestock_production_tonnes_2020.tif", "w", **meta) as dst:
    dst.write(out_arr, 1)